# Track B: reinforcement learning altitude controller

**Integration week, Track B, step 4.** The policy is trained **only on the model we identified from
data** in steps 1-3 (never on `quadcopter_model.p`), then exported to MATLAB and flown on `UAV_model.slx`.

```
            +------------------ identified model (Python) ------------------+
 u[k] ----> | NN  az[k+1] = F(az[k], u[k])  -> integrate -> vz, z            | ---> measurements
            +---------------------------------------------------------------+         |
   ^                                                                                  |
   +---------------  SAC policy  pi(obs)  <----  observation(t, z, vz, az_prev, u_prev, z_ref)
```

| choice | what | why |
|---|---|---|
| plant | NN of `2_identification_nn.ipynb` (2 -> 16 tanh -> 1), **randomised** | the most accurate model (free-run error 0.07 m/s², close to the sensor noise); the tanh Hammerstein model is kept as an *independent* test drone and for the PID baseline |
| algorithm | **SAC** | continuous command, off-policy (sample-efficient), entropy bonus -> smoother and more robust policies than TD3; PPO needs ~10x more samples here |
| observation | error, z, vz, az_prev, u_prev, **integral of error**, z_ref | (z, vz, az_prev, u_prev) is a *complete* state of the identified dynamics (Markov); the integral removes static errors when the real hover thrust differs from the model's |
| reward | tracking + precision + smoothness + ground | see section 2.3 |
| robustness | domain randomisation: model parameters, initial state, missions, process and sensor noise | the policy must survive the model -> simulator gap |

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, torch
torch.set_num_threads(1)
from stable_baselines3 import SAC
from drone_env import (DroneAltitudeEnv, TanhPlant, PIDBaseline, rollout, mission, load_nn_plant,
                       Ts, N_STEPS, Z_GROUND, OBS_NAMES)
from evaluate import (compare, plot_flights,
                      C_SAC, C_PID, C_REF, C_MUTED)
from export_policy import export_for_matlab
os.makedirs("figures", exist_ok=True)
np.set_printoptions(suppress=True, precision=4)

## 1. The plants: our identified models

**Model 1 - NN (training plant)**, from `2_identification_nn.ipynb`: $a_z[k+1] = F(a_z[k], u[k])$,
an MLP 2 -> 16 tanh -> 1 (inputs and output standardised), saved in `models/drone_model_nn.joblib`.
`drone_env.NNPlant` runs it in numpy (much faster than `sklearn.predict` for one sample at a time).

**Model 2 - tanh Hammerstein (test drone + PID)**, from `1_identification_linear.ipynb`, model C:
$a_z[k+1] = a\,a_z[k] + b\tanh(\beta u[k]) + c$ with $a=0.7804,\ b=6.4373,\ c=-2.1545,\ \beta=0.9$.
In physical form: $\tau = -T_s/\ln a \approx 0.20$ s, max gain $K=b/(1-a)\approx 29.3$ m/s², offset
$c/(1-a) = -g$. Hover command: $u_h = \operatorname{atanh}(g/K)/\beta \approx 0.39$.
It is *not* used for training: it plays the role of "another drone" to check that the policy transfers
to a model it has never seen - the same question as model -> `UAV_model.slx`. Its closed-form inverse
is also what the PID baseline uses (feedback linearisation).

In [ ]:
nn_p = load_nn_plant()        # models/drone_model_nn.joblib: the training plant
tanh_p = TanhPlant()          # the test drone (never seen in training)
print(f"tanh model: tau = {tanh_p.tau:.3f} s   K = {tanh_p.K:.2f} m/s^2   g0 = {tanh_p.g0:.3f} m/s^2   hover u = {tanh_p.hover_u():.3f}")
print(f"NN model  : az at rest with u = 0 -> {nn_p.reset(0.0):.3f} m/s^2   max climb {nn_p.reset(1.0):.1f}   max descent {nn_p.reset(-1.0):.1f} m/s^2")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ug = np.linspace(-1, 1, 41)
plants = [(nn_p, C_SAC, '-', 'NN (training)'), (tanh_p, C_PID, '--', 'tanh (test drone)')]
for p, c, ls, n in plants:
    ax[0].plot(ug, [p.reset(u) for u in ug], ls, color=c, label=n)
ax[0].axhline(0, color=C_MUTED, lw=.8); ax[0].set_xlabel('u'); ax[0].set_ylabel('steady-state az [m/s$^2$]')
ax[0].legend(frameon=False); ax[0].set_title('static characteristic', loc='left')
tt = np.arange(30) * Ts
for p, c, ls, n in plants:
    p.reset(0.387); ax[1].plot(tt, [p.step(0.8) for _ in tt], ls, color=c, label=n)
ax[1].set_xlabel('t [s]'); ax[1].set_ylabel('az [m/s$^2$]'); ax[1].set_title('step u: 0.39 -> 0.8', loc='left')
ax[1].legend(frameon=False); plt.tight_layout(); plt.show()

## 2. The environment

### 2.1 Same measurements, same timing as Simulink
`EINT_RL_closed_loop.slx` (only the loop wiring was read, not the drone): `z`, `vz` go through a
zero-order hold, `az` and `u` through **Unit Delays** (so `az_prev` is **0** at $k=0$), the single
command `u` drives the 4 motors. `DroneAltitudeEnv` reproduces exactly this, plus:

* **start**: like every recorded experiment, motors off at $t=0$, so $a_z = -g$: the drone *falls*
  until the motors spin up ($\tau = 0.2$ s). A controller that waits loses ~25 cm (the PID below).
* **kinematics**: $v_z$, $z$ are integrated exactly assuming $a_z$ linear between samples.
* **mission** (Figure 1): 0 m on the platform until 1 s, 0.5 m until 5 s, then -0.25 m (ground).
  Thresholds shifted by $T_s/2$ so Python and Simulink never disagree on the sample at exactly 1 s / 5 s.

### 2.2 Observation (identical in `rl_controller.m`)
$$o = [\,e,\ z,\ v_z/2,\ a_{z,prev}/10,\ u_{prev},\ I/I_{max},\ z_{ref}\,],\qquad e=z_{ref}-z,\quad
I_k = \mathrm{clip}(I_{k-1}+T_s e_k,\ \pm 0.5)$$
$(z, v_z, a_{z,prev}, u_{prev})$ is a full state of the identified model: with the delay,
$a_z[k] = F(a_{z,prev}, u_{prev})$ is computable, so the problem is Markov.

### 2.3 Reward (per step, $T_s = 0.05$ s)
$$r = 1 - |e| - 0.5\,(1-e^{-(e/0.05)^2}) - 0.05\,v_z^2 - 0.5\,\Delta u^2 - 10\,\max(0, -0.25 - z)$$
- $1-|e|$: alive bonus + tracking ($r>0$ in normal flight, so crashing is never attractive);
- the Gaussian term rewards *precision* (< 5 cm), which $|e|$ alone barely does;
- $\Delta u^2$: smooth command - essential with sensor noise (otherwise SAC is bang-bang);
- going **under the ground** is heavily penalised: landing must not overshoot;
- termination with -50 if the drone leaves the envelope ($z<-0.5$, $z>3$ or $|v_z|>6$).

### 2.4 Domain randomisation (training only)
| what | range |
|---|---|
| time constant $\tau$ (speed of the motors) | x [0.8, 1.25] |
| thrust gain $K$ | x [0.85, 1.15] |
| gravity / hover offset $g_0$ | ± 0.4 m/s² |
| process noise on $a_z$ | std up to 0.15 m/s² |
| sensor noise $z$ / $v_z$ / $a_z$ | std up to 1 cm / 0.1 m/s / 0.6 m/s² |
| missions | 40 % Figure 1, 60 % random staircases in [-0.25, 0.8] m |
| initial state | 70 % Simulink start (25 % of them with motors partly spun up), 30 % random flight |

In [ ]:
env = DroneAltitudeEnv(plant=nn_p)
obs, info = env.reset(seed=0)
print('observation:', dict(zip(OBS_NAMES, obs.round(3))))
print('measurements:', {k: round(v, 3) for k, v in info['m'].items()})
o = rollout(env, lambda obs, m: PIDBaseline()(m))
print(f"sanity: z_ref levels on the mission: {sorted(set(np.round(o['z_ref'], 2)))}")

### 2.5 Linear baseline
A PID with **feedback linearisation**: $a_{cmd} = K_p e - K_d v_z + K_i\!\int e$, then
$u = f^{-1}(a_{cmd})$ through the identified tanh map (so the gain is the same at any thrust).
Gains grid-searched on the tanh model under the constraint *never go under the ground*:
$K_p=12,\ K_d=8,\ K_i=0.5$. It uses exactly the measurements the policy has.

## 3. Training (SAC)

Same code as `train_sac.py` (300 k steps = ~4 h of simulated flight; run several seeds in parallel
with `python train_sac.py 300000 <seed>` and keep the best).
`EvalCallback` keeps the checkpoint with the best return on **perturbed models + noise** on the
Figure 1 mission, not the best on the nominal model. Follow it with `tensorboard --logdir runs/tb`.
`TRAIN = False` (default) loads the final agent `models/sac_nn.zip` (300 k steps on the NN plant, seed 0).
`TRAIN = True` retrains it and overwrites `models/sac_nn.zip`.

In [ ]:
TRAIN = False
STEPS = 300_000
if TRAIN:
    import shutil
    from stable_baselines3.common.monitor import Monitor
    from stable_baselines3.common.callbacks import EvalCallback
    train_env = Monitor(DroneAltitudeEnv(plant=nn_p, randomize=True))
    eval_env = Monitor(DroneAltitudeEnv(plant=nn_p, randomize=True, fig1_prob=1.0))
    agent = SAC("MlpPolicy", train_env, seed=0, learning_rate=3e-4, buffer_size=300_000,
                batch_size=256, gamma=0.98, tau=0.005, learning_starts=5_000,
                policy_kwargs=dict(net_arch=[128, 128]), tensorboard_log="runs/tb")
    agent.learn(STEPS, tb_log_name="sac_nn_s0", callback=EvalCallback(eval_env, n_eval_episodes=20, eval_freq=10_000,
                                               best_model_save_path="runs/sac_nn_s0", log_path="runs/sac_nn_s0",
                                               deterministic=True))
    shutil.copy("runs/sac_nn_s0/best_model.zip", "models/sac_nn.zip")
agent = SAC.load("models/sac_nn.zip", device="cpu")
print(agent.policy.actor)

In [ ]:
# learning curves (evaluation return on perturbed models + noise, Figure 1 mission)
plt.figure(figsize=(8, 3))
for f, c, lab in (('runs/sac_nn_s0/evaluations.npz', C_SAC, 'trained on NN (final)'),
                  ('runs/sac_s0/evaluations.npz', C_PID, 'trained on tanh')):
    if os.path.isfile(f):
        d = np.load(f); m, sd = d['results'].mean(1), d['results'].std(1)
        plt.plot(d['timesteps'] / 1e3, m, color=c, label=lab)
        plt.fill_between(d['timesteps'] / 1e3, m - sd, m + sd, color=c, alpha=.12, lw=0)
plt.legend(frameon=False)
plt.xlabel('training steps [k]'); plt.ylabel('evaluation return')
plt.title('SAC learning curves', loc='left'); plt.tight_layout(); plt.show()

## 4. Evaluation

Two flights, SAC against the PID: on the **training model** (the NN, nominal) and on the **tanh model**, which the policy has never seen: the same question as model -> simulator. Robustness to sensor noise is then checked on the real simulator, in MATLAB (`g_noise`).

In [ ]:
cases = [('NN', compare(agent, plant=nn_p), 'NN model (training model, nominal)'),
         ('tanh', compare(agent, plant=tanh_p), 'tanh model (never seen)')]
plot_flights([c[1] for c in cases], [c[2] for c in cases], fname='figures/flights.png'); plt.show()
for name, r, _ in cases:
    for c in ('SAC', 'PID'):
        o = r[c]
        print(f"{name:9s} {c}: RMSE {o['rmse']:.3f} m   min z {o['min_z']:+.3f} m   "
              f"takeoff dip {o['z'][o['t'] < 1].min():+.3f} m   |du| rms {o['du_rms']:.3f}")

## 5. Export to MATLAB

`export_for_matlab` writes, into `../rl_interface/` (where `EINT_RL_eval.m` looks for them):
- `policy.mat`: the actor, 7 -> 128 -> 128 -> 1, ReLU, tanh output;
- `traj_model.mat`: the Figure 1 mission flown on the NN model, without noise.

`mission()` and `observation()` are already written in `rl_interface/rl_controller.m` (same code as
`drone_env.py`). The replay check of `EINT_RL_eval.m` (`check_controller`) verifies that MATLAB
computes the same observation and the same command as Python.

In MATLAB: open `rl_interface/`, run `EINT_RL_eval` with `g_noise = 0`, then `0.1`, then higher.

In [ ]:
here = os.getcwd()
os.chdir(os.path.join("..", "rl_interface"))     # export_for_matlab writes into the current folder
try:
    export_for_matlab(agent, DroneAltitudeEnv(plant=nn_p))
finally:
    os.chdir(here)

## 6. Critical analysis

* **Where RL can win.** The PID structurally cannot anticipate: the motors start at $a_z=-g$, so a
  controller that waits for an error loses ~25 cm at take-off (PID: min z ≈ -0.30 m), while full
  thrust from the first sample limits the dip to a few cm. The policy also sees `az_prev` and `u_prev`,
  i.e. the thrust lag, so it can brake *before* the ground. Check both on the plots of section 4.
* **Where it costs.** The policy is a black box: no stability margin, no guarantee outside the
  randomised ranges. The tanh flight of section 4 is the honest check of how it transfers. If the
  simulator's drone differs in *structure* (extra delay, asymmetric motors), retraining with that
  in the randomisation is the fix, not tuning the reward.
* **Sensor noise.** The command smoothness term ($\Delta u^2$) is what keeps the policy from
  chattering. If, at `g_noise = 0.1`, `u` chatters in MATLAB, raise the noise ranges of section 2.4
  and the $\Delta u$ weight, and retrain.
* **What to report if MATLAB disagrees with Python**: the check (`check_controller`) proves the
  controller code is identical, so any difference comes from the **model** (model-reality gap) - compare
  `z` from Simulink with `traj_model.mat` at `g_noise = 0` to measure it.
* **Assumptions to verify in Simulink**: motors off at $t=0$ ($a_z(0)=-g$, as in the data); no
  physical ground (the drone hovers at -0.25 m). The `rl_controller` integral assumes one call per
  step (`Ts = 0.05`), which is the case with the ZOH / Unit Delay wiring of `EINT_RL_closed_loop.slx`.